In [1]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceBgeEmbeddings
import numpy as np
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

In [2]:
loader = PyPDFDirectoryLoader("./us_census")
documents = loader.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
final_documents = text_splitter.split_documents(documents)
final_documents[0]

Document(metadata={'producer': 'Adobe PDF Library 17.0', 'creator': 'Adobe InDesign 18.2 (Windows)', 'creationdate': '2023-09-09T07:52:17-04:00', 'author': 'U.S. Census Bureau', 'keywords': 'acsbr-015', 'moddate': '2023-09-12T14:44:47+01:00', 'title': 'Health Insurance Coverage Status and Type by Geography: 2021 and 2022', 'trapped': '/false', 'source': 'us_census\\acsbr-015.pdf', 'total_pages': 18, 'page': 0, 'page_label': '1'}, page_content='Health Insurance Coverage Status and Type \nby Geography: 2021 and 2022\nAmerican Community Survey Briefs\nACSBR-015\nIssued September 2023\nDouglas Conway and Breauna Branch\nINTRODUCTION\nDemographic shifts as well as economic and govern-\nment policy changes can affect people’s access to')

In [3]:
len(final_documents)

998

In [4]:
##Embedding Using HuggingFace
huggingface_embeddings = HuggingFaceBgeEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    model_kwargs = {"device":"cpu"},
    encode_kwargs={"normalize_embeddings":True}
)

C:\Users\pande\AppData\Local\Temp\ipykernel_27076\3531129591.py:2: LangChainDeprecationWarning: The class `HuggingFaceBgeEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  huggingface_embeddings = HuggingFaceBgeEmbeddings(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:
import numpy as np
np.array(huggingface_embeddings.embed_query(final_documents[0].page_content))

array([-0.01304135, -0.00212929, -0.02641384,  0.03565067,  0.02980655,
        0.04708725,  0.00126449,  0.01503202, -0.07432815, -0.02466967,
        0.06501861,  0.02952588, -0.06186428,  0.01675615, -0.00839763,
        0.00778627, -0.00941912, -0.00913801,  0.02576545,  0.05559567,
       -0.04097351,  0.01464152, -0.05042764, -0.03896266,  0.02162804,
       -0.02777419,  0.00113016, -0.00819398, -0.06824131, -0.14591628,
        0.02289139,  0.03154322, -0.04227985, -0.03389408,  0.02062804,
       -0.02617834,  0.00290701,  0.03869241,  0.03532048,  0.03521501,
       -0.02316842,  0.00982881,  0.02808383, -0.00969876, -0.04232314,
        0.02295368, -0.0219514 ,  0.03778228, -0.02935382, -0.04722689,
        0.01430155, -0.04056733,  0.04078627,  0.06784478,  0.06011226,
       -0.03884673,  0.02182321, -0.00346487, -0.02669143,  0.04154936,
        0.04588628, -0.02176328, -0.23292492,  0.07723476, -0.00648733,
        0.08012363,  0.00253782, -0.00247521, -0.02089968, -0.04

In [6]:
## Step 1: Create FAISS VectorStore from documents + embeddings
vectorstore = FAISS.from_documents(final_documents, huggingface_embeddings)
print('VectorStore created with', vectorstore.index.ntotal, 'vectors')

VectorStore created with 998 vectors


In [7]:
## Step 2: Quick Similarity Search Test
query = 'What is health insurance coverage?'
relevant_docs = vectorstore.similarity_search(query, k=3)

for i, doc in enumerate(relevant_docs):
    print(f'--- Document {i+1} ---')
    print(doc.page_content[:300])
    print()

--- Document 1 ---
2 U.S. Census Bureau
WHAT IS HEALTH INSURANCE COVERAGE?
This brief presents state-level estimates of health insurance coverage 
using data from the American Community Survey (ACS). The  
U.S. Census Bureau conducts the ACS throughout the year; the

--- Document 2 ---
private health insurance as a plan provided through an employer 
or a union, coverage purchased directly by an individual from an 
insurance company or through an exchange (such as healthcare.
gov), or coverage through TRICARE. Public insurance coverage

--- Document 3 ---
insurance at time of interview or if they only had coverage through 
the Indian Health Service (IHS), as IHS coverage is not considered 
comprehensive.
* Comprehensive health insurance covers basic health care needs. This definition



In [8]:
## Step 3: Create Retriever
retriever = vectorstore.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 2}
)

In [9]:
## Step 4: Define Prompt Template
prompt_template = """
Use the following pieces of context to answer the question at the end.
If you don't know the answer, just say that you don't know do not make up an answer.
Keep your answer concise (3-4 sentences max).

Context:
{context}

Question: {question}

Helpful Answer:"""

prompt = PromptTemplate(
    template=prompt_template,
    input_variables=['context', 'question']
)
print(prompt)

input_variables=['context', 'question'] input_types={} partial_variables={} template="\nUse the following pieces of context to answer the question at the end.\nIf you don't know the answer, just say that you don't know do not make up an answer.\nKeep your answer concise (3-4 sentences max).\n\nContext:\n{context}\n\nQuestion: {question}\n\nHelpful Answer:"


In [10]:
## Step 5: Load HuggingFace LLM via Transformers pipeline
from langchain_community.llms import HuggingFacePipeline
from transformers import pipeline

hf_pipeline = pipeline(
    'text2text-generation',
    model='google/flan-t5-base',
    max_new_tokens=128,
    temperature=0.2,
    device=-1,
    repetition_penalty=1.5,
    no_repeat_ngram_size=3,
    truncation=True,
    
)

llm = HuggingFacePipeline(pipeline=hf_pipeline)
print('LLM loaded:', llm)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The `Text2TextGenerationPipeline` is deprecated and no longer maintained. For most purposes, we recommend using newer models with causal pipelines like `TextGenerationPipeline` or `ImageTextToTextPipeline`.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


LLM loaded: HuggingFacePipeline
Params: {'model_id': 'gpt2', 'model_kwargs': None, 'pipeline_kwargs': None}


C:\Users\pande\AppData\Local\Temp\ipykernel_27076\105808697.py:17: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(pipeline=hf_pipeline)


In [11]:
## Step 6: Helper format retrieved docs into a single string (with truncation)
def format_docs(docs, max_chars=1500):
    text = "\n\n".join(doc.page_content for doc in docs)
    return text[:max_chars]

## Step 7: Build the RAG Chain
rag_chain = (
    {'context': retriever | format_docs, 'question': RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print('RAG chain ready!')

RAG chain ready!


In [12]:
## Step 8: Ask a Question!
query = 'What is health insurance coverage status in 2022?'
response = rag_chain.invoke(query)
print('Answer:', response)

Answer: fühlt fühlt fühlt weil weil weil universitaire universitaire universitaire weil weildeshalbdeshalbdeshalblbstverständlichlbstverständlichlbstverständlichAufAufAuf Sherlock Sherlock Sherlock également également égalementußerdemußerdemußerdem weil weil baie baie baie universitaire universitaire Liege Liege Liege Bhutan Bhutan BhutanByronByronByronmöbelmöbelmöbel impun impun impun dés dés dés impun impun nascut nascut weil weilScienceScienceSciencegefühlgefühl Berufs Berufs BerufsAvemAvemAvem universitaire universitaire réalisée réalisée réalisée baie baieUniversitéUniversitéUniversité Jamaica Jamaica Jamaica Bermuda Bermuda BermudaByronByron universitaire universitaireextérieurextérieur weil weilwarumwarumwarum weilschuheschuheschuhe wollte wollte wollte universitaire Engel Engel Engel weil weil trotzdem trotzdem trotzdemdeshalbdeshalb daher daherdeshalbdeshalb universitaire universitaire baie baiemeister clientèle impun impun
